# Object-centric data extraction: smoke test (PickCube-v1, 5 demos, RGB-D)

Replays five official ManiSkill motion-planning demonstrations on `physx_cpu`, records RGB, depth, cube pose
(simulator ground truth), end-effector pose, joint positions/velocities, expert actions and success, validates
them, and saves an HDF5 file. All logic lives in `scripts/object_centric_smoke_test.py` and `src/data/object_centric.py`.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. The simulation runs on CPU (`physx_cpu`), but the
camera renderer needs a GPU Vulkan driver; a CPU-only runtime cannot render images. No training happens here.

## 1. Code

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/Shahriarmsakib/robot-visual-imitation-maniskill.git"
CODE = "/content/robot-visual-imitation-maniskill"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. Install

In [ ]:
# Vulkan driver files + project packages (keeps Colab's own PyTorch)
!bash scripts/setup_linux_gpu.sh

## 3. Demonstrations

In [ ]:
# Official PickCube-v1 motion-planning demos (read-only input; not modified)
!python -m mani_skill.utils.download_demo PickCube-v1 -o data/demos

## 4. Extract, validate, save

In [ ]:
!python scripts/object_centric_smoke_test.py --num-demos 5

## 5. Visualisation and validation summary

In [ ]:
import json
from IPython.display import Image, display
display(Image("results/object_centric_smoke/smoke_frames.png"))
report = json.load(open("results/object_centric_smoke/validation.json"))
for c in report["checks"]:
    print(("PASS " if c["passed"] else "FAIL ") + c["check"] + ": " + c["detail"])

## 6. Dataset layout

In [ ]:
# How to read the saved dataset
import h5py
with h5py.File("data/object_centric/pickcube_5demos_rgbd.h5", "r") as f:
    print(f.attrs["alignment"])
    ep = f["episodes/ep_000"]
    for name, item in ep.items():
        if isinstance(item, h5py.Dataset):
            print(f"  {name}: {item.shape} {item.dtype}")
        else:
            for k, v in item.items():
                print(f"  {name}/{k}: {v.shape} {v.dtype}")